# Validating data with Pandera

## Chosen topic

Data validation with a focus on **Pandera**, applied to **email campaign data**.

The goal is to learn how to define and use schemas in Pandera, show which rows pass and which are rejected, as well as why. 

Reliable data is a fundamental prerequisite for analyses and predictive models, which connects directly to my internship where data quality is part of the work.


## Definition

Pandera is an open source project that allows us to perform data validation on dataframe-like objects.
The goal is to **make data processing pipelines more readable and robust**. 

Pandera makes it possible to:  
- Define a schema once and use it to validate different dataframes.  
- Check the types and properties of columns in a `pd.DataFrame` or values in a `pd.Series`.
- Parse data to standardize the preprocessing steps needed to produce valid data.
- Integrate with existing data analysis/processing pipelines via function decorators.  
- Define dataframe models.   
- Lazily validate dataframes so that all validation rules are executed before raising an error.  


## Starting questions

1. How do I define a schema and what does it check?  
2. What constraints can I put on columns beyond dtype? 
3. What happens when validation fails?
4. How does Pandera fit into a pipeline?

Summary and answers at the bottom of this notebook.

In [1]:
from importlib.metadata import version
import pandas as pd
import pandera.pandas as pa
from pandera.typing.pandas import Series

print(version('pandera'))

0.33.1


A schema in Pandera is like a contract that describes the **expected structure and properties of the data**.  
When validating a dataframe against a schema, Pandera checks that every aspect matches the specifications.

**Schemas consist in column definition**, each with data type and optional constraints.

Validation checks **three key aspects**:
- **Structure** (are all the required columns there?)
- **Types** (is each column the correct datatype?)
- **Values** (do values satisfy all the constraints?)

Pandera offers two main approaches: 
- Object-based API: `DataFrameSchema`
- Class-based API: `DataFrameModel`

In [2]:
# Create basic email campaign data
data = {
    'subject_line': ['Spring Sale', 'Newsletter #42', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': [1200, 1150, 1200, 1300, 1150],
    'sent_at': ['2026-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [340, 210, 340, 520, 190],
    'cost_per_email': [0.05, 0.08, 0.05, 0.12, 0.08],
}

df = pd.DataFrame(data)
df['sent_at'] = pd.to_datetime(df['sent_at'])

### Basic type validation with DataFrameSchema

Pandera documentation:   
DataFrameSchema - https://pandera.readthedocs.io/en/stable/dataframe_schemas.html  
dtypes - https://pandera.readthedocs.io/en/stable/dtypes.html and https://pandera.readthedocs.io/en/stable/reference/dtypes.html#api-dtypes

In [3]:
# Creating a schema that validates structure and basic types
basic_schema = pa.DataFrameSchema({
    'subject_line': pa.Column(str),
    'recipients': pa.Column(int),
    'sent_at': pa.Column(pa.DateTime),
    'opens': pa.Column(int),
    'cost_per_email': pa.Column(float)
})

# Validating the df - the method returns the df silently when it succeds
validated_df = basic_schema.validate(df)

### Checks and categorical contraints

Checks allow you to specify properties about dataframes, columns, indexes, and series objects, which are applied after data type validation/coercion.  
Multiple checks can be applied to the same column and check objects accept functions as a required argument.

Pandera documentation. Checks - https://pandera.readthedocs.io/en/stable/checks.html

In [4]:
# Built-in checks (operating on pd.series)
enhanced_schema = pa.DataFrameSchema({
    'subject_line': pa.Column(str, pa.Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])),
    'recipients': pa.Column(int, pa.Check.greater_than_or_equal_to(0)),
    'sent_at': pa.Column(pa.DateTime, pa.Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01'))),
    'opens': pa.Column(int, pa.Check.greater_than_or_equal_to(0)),
    'cost_per_email': pa.Column(float, pa.Check.in_range(0, 0.2))
})

validated_df = enhanced_schema.validate(df)

validated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_line    5 non-null      str           
 1   recipients      5 non-null      int64         
 2   sent_at         5 non-null      datetime64[us]
 3   opens           5 non-null      int64         
 4   cost_per_email  5 non-null      float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(1)
memory usage: 332.0 bytes


### Coercing and allowing missing values

Pandera is primarily a validation library that checks data without changing anything about the dataframe itself.

However, in many cases its useful to parse the data to ensure the values reflect what is specified in the schema. Pandera's built in coercion allows to do dtype casting specifically by passing in the `coerce=True`.

Pandera treats all columns as non-nullable, so to allow missing values in a column set `nullable=True`.

In the checking phase, the null values are ignored. But if it is important that they are included in the check, then specify `Check(..., ignore_na=False)` when defining a check.

Pandera documentation:  
Coercion - https://pandera.readthedocs.io/en/stable/dtype_validation.html  
Nulls values - https://pandera.readthedocs.io/en/stable/checks.html

In [5]:
data_to_coerce = {
    'subject_line': [ 'Spring Sale', 'Newsletter #42', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': ['1200', 1150.0, 1200, 1300, 1150],
    'sent_at': [None, '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [None, 210, 340.0, 520, 190],
    'cost_per_email': [None, 0.08, 0.05, 0.12, 1],
}

df_to_coerce = pd.DataFrame(data_to_coerce)

In [6]:
schema_with_coerce = pa.DataFrameSchema({
    'subject_line': pa.Column(str, pa.Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])),
    'recipients': pa.Column(int, pa.Check.in_range(1, 5000)),
    'sent_at': pa.Column(pa.DateTime, pa.Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01')), nullable=True),
    'opens': pa.Column(pd.Int32Dtype, pa.Check.in_range(0, 5000), nullable=True),
    'cost_per_email': pa.Column(float, pa.Check.in_range(0.0, 1.0), nullable=True)
    },
    coerce=True 
)

coerced_df = schema_with_coerce.validate(df_to_coerce)

coerced_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_line    5 non-null      str           
 1   recipients      5 non-null      int64         
 2   sent_at         4 non-null      datetime64[ns]
 3   opens           4 non-null      Int32         
 4   cost_per_email  4 non-null      float64       
dtypes: Int32(1), datetime64[ns](1), float64(1), int64(1), str(1)
memory usage: 317.0 bytes


### Cross-Column Validation

The previous examples show column checks that only look at one column. Sometimes, though, a rule depends on how two or more columns relate to each other.  
For that Pandera supports **schema-level checks**: pass them via the `checks=` argument on `pa.DataFrameSchema` itself.  
The check function receives the whole DataFrame, so it can compare columns directly and the failures are reported with less granularity than column-level ones.

Statology guide - https://www.statology.org/data-validation-in-python-with-pandera-a-practical-introduction/

In [7]:
# to check opens <= recipients
def check_opens_vs_recipients(df: pd.DataFrame) -> pd.Series:
    return (df['opens'] <= df['recipients'])

full_schema = pa.DataFrameSchema({
    'subject_line': pa.Column(str, pa.Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch']), nullable=True),
    'recipients': pa.Column(int, pa.Check.in_range(1, 5000)),
    'sent_at': pa.Column(pa.DateTime, pa.Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01')), nullable=True),
    'opens': pa.Column(pd.Int16Dtype, pa.Check.in_range(0, 5000), nullable=True),
    'cost_per_email': pa.Column(float, pa.Check.in_range(0.0, 1.0), nullable=True)
    },
    checks=pa.Check(check_opens_vs_recipients, error='opens <= recipients'),
    coerce=True
    )

validated_df = full_schema.validate(df_to_coerce)

validated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_line    5 non-null      str           
 1   recipients      5 non-null      int64         
 2   sent_at         4 non-null      datetime64[ns]
 3   opens           4 non-null      Int16         
 4   cost_per_email  4 non-null      float64       
dtypes: Int16(1), datetime64[ns](1), float64(1), int64(1), str(1)
memory usage: 307.0 bytes


### Validation failure

By default, when one of the validation requirements is broken, a `SchemaError` is raised and the validation stopped.

This behavior is fine for simple tasks, but in more complex situations we want to collect all the errors at once. Pandera can provide an **error report** that includes detailes error information. 

To create an error report with pandas, you must specify `lazy=True` to allow all errors to be aggregated and raised together as a `SchemaErrors`.

Pandera documentation:  
Error report - https://pandera.readthedocs.io/en/stable/error_report.html  
Lazy validation - https://pandera.readthedocs.io/en/stable/lazy_validation.html

Statology guide - https://www.statology.org/data-validation-in-python-with-pandera-a-practical-introduction/

In [8]:
bad_data = {
    'subject_line': [ 'Spring Sale', 'Newsletter', 'Spring Sale', 'Product Launch', None],
    'recipients': ['1200', 1150.0, 1200, 1300, 1150],
    'sent_at': [None, '2026-03-15', '2026-03-01', '2026-04-02', '2025-12-30'],
    'opens': [1220, 210, 340.0, 520, 190],
    'cost_per_email': [None, 0.08, 0.05, 0.12, 1],
}

bad_df = pd.DataFrame(bad_data)

In [9]:
# Running the bad_data df through the full schema to analyze the error

# validate_bad_data = full_schema.validate(bad_df, lazy=True)
# Commented out as it raises SchemaError and interrupts the notebook. Uncomment and run to show the message and checks that were broken.

Wrapping the call to validate into a `try/except` block catches Pandera's error types and allows us to handle validation failure more gracefully.  
The `SchemaErrors` that is raised carries `.failure_cases` which is a **df listing of which columns, indexes, values failed and why**.  

Then you can decide to log them for a human to review, quarantine/drop just those rows and keep processing the rest, or route them back for cleaning and re-validation, but that decision should be explicit and visible.

In [10]:
# Handling failure more gracefully
try:
    validated_bad_df = full_schema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print('Schema errors and failure cases:')
    display(exc.failure_cases)
    print('\nDataFrame object that failed validation:')
    display(exc.data)

Schema errors and failure cases:


,schema_context,column,check,check_number,failure_case,index
2,DataFrameSchema,subject_line,opens <= recipients,0,Spring Sale,0
3,DataFrameSchema,recipients,opens <= recipients,0,1200,0
4,DataFrameSchema,opens,opens <= recipients,0,1220,0
0,Column,subject_line,"isin(['Spring Sale', 'Newsletter #42', 'Produc...",0,Newsletter,1
1,Column,sent_at,greater_than_or_equal_to(2026-01-01 00:00:00),0,2025-12-30 00:00:00,4



DataFrame object that failed validation:


,subject_line,recipients,sent_at,opens,cost_per_email
0,Spring Sale,1200,NaT,1220,NaN
1,Newsletter,1150,2026-03-15,210,0.08
2,Spring Sale,1200,2026-03-01,340,0.05
3,Product Launch,1300,2026-04-02,520,0.12
4,NaN,1150,2025-12-30,190,1.00


### Showing which rows pass and which are rejected

It is possible to directly drop the invalid rows by adding `drop_invalid_rows=True` to the schema.  

Silently dropping rows can mask data quality issues, so a better approach is to log the invalid rows before removing them.

Pandera documentation. Dropping invalid rows - https://pandera.readthedocs.io/en/stable/drop_invalid_rows.html  
Python Data Bench guide - https://pythondatabench.com/article/data-validation-python-pandera-practical-guide

In [11]:
invalid_data = {
    'subject_line': [ 'Spring Sale', 'Newsletter', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': [1200, 1150.0, 1200, 1300, 1150],
    'sent_at': ['2026-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2025-12-30'],
    'opens': [1220, 210, 340.0, 520, 190],
    'cost_per_email': [0.1, 0.08, 0.05, 0.12, 1],
}

invalid_df = pd.DataFrame(invalid_data)

In [12]:
try:
    validated_invalid_df = full_schema.validate(invalid_df, lazy=True)
    print('All rows passed.')
except pa.errors.SchemaErrors as exc:
    failed_idx = exc.failure_cases['index'].dropna().unique()
    passed_df = invalid_df.drop(index=failed_idx)
    rejected_df = invalid_df.loc[failed_idx]

    # One row per rejected record, with all its failure reasons collapsed into a single cell
    reasons = (
        exc.failure_cases.dropna(subset=['index'])
        .groupby('index')
        .apply(lambda g: '; '.join(f"{r.column}: {r.check} (got {r.failure_case})" for r in g.itertuples()))
    )

    combined = rejected_df.copy()
    combined['rejection_reason'] = combined.index.map(reasons)

    print('Passed rows:')
    display(passed_df)

    print('Rejected rows with reasons:')
    display(combined)

    #combined.to_csv('rejected_rows_log.csv', index=False)

Passed rows:


,subject_line,recipients,sent_at,opens,cost_per_email
2,Spring Sale,1200.0,2026-03-01,340.0,0.05
3,Product Launch,1300.0,2026-04-02,520.0,0.12


Rejected rows with reasons:


,subject_line,recipients,sent_at,opens,cost_per_email,rejection_reason
0,Spring Sale,1200.0,2026-03-01,1220.0,0.10,subject_line: opens <= recipients (got Spring ...
1,Newsletter,1150.0,2026-03-15,210.0,0.08,"subject_line: isin(['Spring Sale', 'Newsletter..."
4,Newsletter #42,1150.0,2025-12-30,190.0,1.00,sent_at: greater_than_or_equal_to(2026-01-01 0...


### Basic type validation with DataFrameModel

`DataFrameModel` is a class-based API where each column becomes a typed class attribute. It is preferred for bigger projects as it scales better. It mirrors the Pydantic pattern in defining schemas as Python classes.

With this short example I wanted to get a general sense of how `DataFrameModel` works. I will keep exploring `DataFrameSchema` as it is the foundation for `DataFrameModel` and for a project this size I'd rather focusing on the basics of the library.

Pandera documentation. DataFrame Models - https://pandera.readthedocs.io/en/stable/dataframe_models.html

In [13]:
class EmailCampaign(pa.DataFrameModel):
    subject_line: Series[str] = pa.Field(isin=['Spring Sale', 'Newsletter #42', 'Product Launch'], coerce=True, nullable=True)
    recipients: Series[int] = pa.Field(ge=0, le=5000, coerce=True)
    sent_at: Series[pa.DateTime] = pa.Field(ge=pd.Timestamp('2026-01-01'), coerce=True, nullable=True)
    opens: Series[int] = pa.Field(ge=0, coerce=True, nullable=True)
    cost_per_email: Series[float] = pa.Field(in_range={'min_value': 0.0, 'max_value': 1}, coerce=True, nullable=True)

    @pa.dataframe_check # Custom check
    def check_opens_vs_recipients(cls, df: pd.DataFrame) -> pd.Series:
        return (df['opens'] <= df['recipients']) 

df_validated_model = EmailCampaign.validate(df_to_coerce, lazy=True)

### Preprocessing with Parsers

Pandera distinguishes between data validation and parsing. Validation is the act of verifying whether data follows some set of constraints, whereas parsing transforms raw data into some desired set of constraints.

The `Parser` abstraction allows you to specify any arbitrary transform that occurs before validation so that you can codify and standardize the preprocessing steps needed to get your raw data into a valid state.

It is useful if the data has formatting issues that would fail in validation (symbols, casing, whitespaces etc.)

Pandera documentation. Parsers - https://pandera.readthedocs.io/en/stable/parsers.html

In [14]:
messy_data = {
    'subject_line': [ ' Spring Sale  ', 'newsletter #42', 'Spring Sale', 'PRODUCT LAUNCH', 'newsletter #42'],
    'recipients': [1200, 1150.0, 1200, 1300, 1150],
    'sent_at': ['2026-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [340, 210, 340.0, 520, -190],
    'cost_per_email': [0.1, 1, '0.05sek', 0.12, 1],
}

messy_df = pd.DataFrame(messy_data)

def clean_cost(s):
    s = s.astype(str).str.replace('sek', '', regex=False)
    return s

messy_schema = pa.DataFrameSchema({
    'subject_line': pa.Column(
        str, 
        parsers=pa.Parser(lambda s: s.str.strip().str.title())
    ),
    'recipients': pa.Column(int, coerce=True),
    'sent_at': pa.Column(pa.DateTime, coerce=True),
    'opens': pa.Column(
        int,
        parsers=pa.Parser(lambda s: s.clip(lower=0)),
        coerce=True),
    'cost_per_email': pa.Column(
        float,
        parsers=pa.Parser(clean_cost),
        coerce=True)
})

parsed_df = messy_schema.validate(messy_df)

display(parsed_df)

,subject_line,recipients,sent_at,opens,cost_per_email
0,Spring Sale,1200,2026-03-01,340,0.10
1,Newsletter #42,1150,2026-03-15,210,1.00
2,Spring Sale,1200,2026-03-01,340,0.05
3,Product Launch,1300,2026-04-02,520,0.12
4,Newsletter #42,1150,2026-03-15,0,1.00


### Pipeline integration

Pandera provides decorators that allow you to embed validation directly into existing function-based data pipelines.  
When an upstream data source changes its schema unexpectedly, the decorated act as contract enforcement points and fails right away with a clear error message instead of producing corrupted output downstream.

The decorators are:  
- `check_input()` - validates input before entering the wrapped function.
- `check_output()` - validates the output of the decorated function.
- `check_types()` - validates inputs and outputs based on Python type annotations.
- `check_io()` - validates inputs and outputs by explicitly passing schema objects as arguments to the decorator.

Pandera documentation. Pipeline integration - https://pandera.readthedocs.io/en/stable/decorators.html  
Python Data Bench. Example of Pandera in ML pipeline - https://pythondatabench.com/article/data-validation-python-pandera-practical-guide

In [15]:
# Basic example on check_input
@pa.check_input(messy_schema)
def compute_open_rate(df):
    df = df.copy()
    df['open_rate'] = round(df['opens'] / df['recipients'], 2)
    return df

pipeline_df = compute_open_rate(messy_df) 
display(pipeline_df)

,subject_line,recipients,sent_at,opens,cost_per_email,open_rate
0,Spring Sale,1200,2026-03-01,340,0.10,0.28
1,Newsletter #42,1150,2026-03-15,210,1.00,0.18
2,Spring Sale,1200,2026-03-01,340,0.05,0.28
3,Product Launch,1300,2026-04-02,520,0.12,0.40
4,Newsletter #42,1150,2026-03-15,0,1.00,0.00


### Summary

This notebook explored Pandera, a schema-based validation library for dataframes. The goal was to learn how schemas are defined, how they catch bad data and how the library fits into a data pipeline.

1. **How do I define a schema and what does it check?**  
A schema is built with `DataFrameSchema` (object-based) or `DataFrameModel` (class-based), mapping each column name to an expected dtype. Calling `.validate()` checks three things: structure (are the right columns present), types (does each column match its declared dtype) and values (do entries satisfy any additional constraints). If everything passes, the method just returns the dataframe silently.

2. **What constraints can I put on columns beyond dtype?**  
Beyond dtype, `pa.Check` adds value-level rules: `isin()` for categorical whitelists, `greater_than_or_equal_to()` / `in_range()` for numeric/date bounds, and custom functions for logic that isn't built in. Constraints can also span multiple columns (like for opens <= recipients) via a schema-level `pa.Check`, and `DataFrameModel` supports the same idea through a `@pa.dataframe_check`-decorated method. Nullability is opt-in (`nullable=True`), and `coerce=True` lets Pandera cast values into the target dtype before checking them. A separate `Parser` step can also clean up formatting issues (whitespace, casing, stray units) before validation runs.

3. **What happens when validation fails?**  
By default, the first broken rule raises a `SchemaError` and stops validation immediately. Passing `lazy=True` instead collects every failure across the whole dataframe and raises a `SchemaErrors` exception, whose `.failure_cases` attribute is a dataframe listing which column, check and value failed, row by row. This makes it possible to catch the exception, split the data into passed vs. rejected rows, attach a human-readable reason to each rejected row and decide explicitly what to do with them (log, drop, send back for cleaning) rather than losing that information.

4. **How does Pandera fit into a pipeline?**  
Pandera provides decorators (`check_input()`, `check_output()`, `check_types()`, `check_io()`) that attach a schema directly to a pipeline function.  
`check_input()`, for example, validates a dataframe the moment it enters a function, so if an upstream source changes shape or values unexpectedly, the pipeline fails fast with a clear error instead of silently propagating bad data downstream. This turns the schema into an enforceable contract at each stage of a pipeline rather than a one-off check run manually.


### Sources:  
Pandera documentation: https://pandera.readthedocs.io/en/stable/  
Statology: https://www.statology.org/data-validation-in-python-with-pandera-a-practical-introduction/  
Python Data Bench: https://pythondatabench.com/article/data-validation-python-pandera-practical-guide  
Medium: https://medium.com/towards-artificial-intelligence/your-model-is-fine-your-data-isnt-dataframe-validation-with-pandera-1552c0daeeaf  